# 04 — Ajuste de hiperparámetros

Randomized search evalúa 30 configuraciones por cada uno de los tres mejores candidatos, con los mismos cinco pliegues agrupados por InChIKey. Cada combinación queda como corrida hija MLflow bajo una corrida padre.

In [1]:
from pathlib import Path
import os
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import mlflow
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from src.datos import load_snapshot
from src.experiments import EXPERIMENT_NAME, configure_tracking, run_nested_random_search
from src.pipeline import make_default_models, make_group_cv, make_search_spaces, model_inputs, valid_smiles_mask
from src.splits import create_or_load_holdout, stratified_compound_sample

frame, metadata = load_snapshot()
valid = valid_smiles_mask(frame)
all_data = frame.loc[valid & frame["inchikey"].notna()].reset_index(drop=True)
train_full, holdout_full, split_metadata = create_or_load_holdout(all_data, source_sha256=metadata["sha256"])
train_sample, sample_metadata = stratified_compound_sample(train_full, max_compounds=12_000, random_state=42)
model_metadata = {**metadata, "source_rows": metadata["rows"], "rows": len(train_sample), "training_sample_sha256": sample_metadata["sha256"], "sampling_method": sample_metadata["method"], "sampling_seed": sample_metadata["random_state"]}
X_train, y_train, groups_train = model_inputs(train_sample)
candidate_table = pd.read_csv(PROJECT_ROOT / "docs" / "candidatos.csv")
top_families = candidate_table.nsmallest(3, "cv_rmse_mean")["family"].tolist()
print({"muestra_estratificada": sample_metadata, "tres_familias_seleccionadas_con_CV": top_families})
cv = make_group_cv(n_splits=5)
models = make_default_models()
search_spaces = make_search_spaces()
configure_tracking()
client = MlflowClient()
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

{'muestra_estratificada': {'rows': 19406, 'compounds': 12000, 'method': 'whole-compound sample stratified by compound-median retention quintile', 'random_state': 42, 'sha256': 'e9b17add3658608f8ec3b37679d79d3127eebfb736fcab73823d659f6014644f'}, 'tres_familias_seleccionadas_con_CV': ['hist_gradient_boosting', 'random_forest', 'mlp']}


In [2]:
tuning_tables = []
for family in top_families:
    existing_runs = client.search_runs(
        [experiment.experiment_id],
        filter_string=(
            f"tags.phase = 'tuning' AND tags.model_family = '{family}' "
            f"AND tags.training_sample_sha256 = '{sample_metadata['sha256']}'"
        ),
        run_view_type=ViewType.ALL,
        max_results=1000,
    )
    completed = {}
    for run in existing_runs:
        if run.info.status != "FINISHED" or "_config_" not in run.info.run_name:
            continue
        try:
            configuration = int(run.info.run_name.rsplit("_", 1)[1])
        except ValueError:
            continue
        if 1 <= configuration <= 30:
            completed.setdefault(configuration, run)

    if set(completed) == set(range(1, 31)):
        family_results = pd.DataFrame([
            {
                "family": family,
                "configuration": configuration,
                **{name: run.data.params[name] for name in search_spaces[family]},
                **{name: run.data.metrics[name] for name in (
                    "cv_rmse_mean", "cv_rmse_std", "train_rmse_mean",
                    "rmse_train_cv_gap", "cv_mae_mean", "cv_fit_seconds_total",
                )},
                "run_id": run.info.run_id,
            }
            for configuration, run in sorted(completed.items())
        ])
        print(f"{family}: se reutilizan 30 configuraciones MLflow ya completadas.")
    else:
        family_results, best_parameters = run_nested_random_search(
            family=family,
            estimator=models[family],
            search_space=search_spaces[family],
            X=X_train,
            y=y_train,
            groups=groups_train,
            cv=cv,
            dataset_metadata=model_metadata,
            seed=42,
            n_iter=30,
        )
    tuning_tables.append(family_results)
tuning_table = pd.concat(tuning_tables, ignore_index=True).sort_values("cv_rmse_mean")
tuning_table.to_csv(PROJECT_ROOT / "docs" / "tuning.csv", index=False)
display(tuning_table.groupby("family").agg(configuraciones=("configuration", "count"), mejor_RMSE_CV=("cv_rmse_mean", "min"), promedio_RMSE_CV=("cv_rmse_mean", "mean"), desviacion_pliegues=("cv_rmse_std", "mean")))
assert tuning_table.groupby("family")["configuration"].nunique().eq(30).all()
print("Tuning completado: 90 configuraciones registradas en MLflow.")

hist_gradient_boosting: se reutilizan 30 configuraciones MLflow ya completadas.


random_forest: se reutilizan 30 configuraciones MLflow ya completadas.


/workspaces/data-science-end-to-end/psets/pset-03/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/workspaces/data-science-end-to-end/psets/pset-03/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/workspaces/data-science-end-to-end/psets/pset-03/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


/workspaces/data-science-end-to-end/psets/pset-03/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


... (119 outputs left out)

/workspaces/data-science-end-to-end/psets/pset-03/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


🏃 View run mlp_config_30 at: http://localhost:5000/#/experiments/1/runs/49344f98ea4648cc87a60c5730b52ebe
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run mlp_tuning_parent at: http://localhost:5000/#/experiments/1/runs/dcc5aee430bc4b21b613e592a374ec3e
🧪 View experiment at: http://localhost:5000/#/experiments/1


,configuraciones,mejor_RMSE_CV,promedio_RMSE_CV,desviacion_pliegues
family,,,,
hist_gradient_boosting,30,2.591868,2.676982,0.230197
mlp,30,2.754353,2.819437,0.215345
random_forest,30,2.573295,2.649234,0.227228


Tuning completado: 90 configuraciones registradas en MLflow.


La búsqueda es aleatoria y reproducible (`seed=42`); los rangos logarítmicos se usan para parámetros de escala amplia como regularización y tasa de aprendizaje. Una diferencia respecto al default menor que la variación entre pliegues no se tratará como mejora concluyente.